# V40 DATA EXCHANGE — FULL GPU ARM (B1 crucible)
Bar: TS-val CE <= **2.2706** (v38e uniform-allocation control @ 13ep).

**Runtime must be T4 GPU** (preselected below — if it connects as CPU, Runtime > Change runtime type > T4).

Run Cell 1, then Cell 2, keep the tab open (~3h). Verdict prints at the end and lands in `/content/v40_report.json`.

In [ ]:
# ==== V40 CELL 1 — setup (T4 GPU) ====
import os, json, subprocess, sys
os.makedirs("/root/.kaggle", exist_ok=True)

# ---- 1) creds: upload (any filename), or CANCEL and paste the key string ----
if os.path.exists("/root/.kaggle/kaggle.json"):
    print("[creds] already installed")
else:
    tok = None
    try:
        from google.colab import files
        up = files.upload()          # CANCEL is fine -> manual fallback
        if up:
            k = next(iter(up)); tok = json.loads(up[k].decode())
            print(f"[creds] loaded from upload '{k}'")
    except Exception as e:
        print("[creds] upload failed:", str(e)[:120])
    if tok is None:
        import getpass
        tok = {"username": "albanchigozirim",
               "key": getpass.getpass("paste ONLY the key string from kaggle.json (no quotes): ")}
        print("[creds] manual key entry")
    assert tok.get("username") and tok.get("key"), "invalid token (need username+key)"
    open("/root/.kaggle/kaggle.json", "w").write(json.dumps(tok))
os.chmod("/root/.kaggle/kaggle.json", 0o600)
tok = json.load(open("/root/.kaggle/kaggle.json"))
os.environ["KAGGLE_USERNAME"] = tok["username"]; os.environ["KAGGLE_KEY"] = tok["key"]
print(f"[creds] user={tok['username']} key=...{tok['key'][-4:]} (verify this matches your file)")

# ---- 2) creds SELF-TEST before any download ----
r = subprocess.run([sys.executable, "-m", "kaggle", "datasets", "list", "--mine", "--page-size", "2"],
                   capture_output=True, text=True)
print("[creds-test] rc", r.returncode, "|", (r.stdout or r.stderr).strip()[:300])
assert r.returncode == 0, "CREDS SELF-TEST FAILED — the token is wrong; rerun cell 1 and re-paste"

# ---- 3) fetch all 5 datasets via kagglehub (clear errors, flat into /content/data) ----
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle", "kagglehub"], check=True)
import kagglehub, shutil
os.makedirs("/content/data", exist_ok=True)
for slug in ["tinystories-gate2-data", "tinystories-gate3-data", "ag-news-v1",
             "curriculum-v37-ckpt", "v40-exchange-src"]:
    p = kagglehub.dataset_download(f"albanchigozirim/{slug}")
    shutil.copytree(p, "/content/data", dirs_exist_ok=True)
    print("[fetch]", slug, "ok")
os.environ["V40_DATA"] = "/content/data"

# ---- 4) ANCHOR + ASSERT: exchange plumbing must be present before anything runs ----
KERNEL = "/content/data/kernel.py"
assert os.path.exists(KERNEL), "kernel.py missing from v40-exchange-src dataset"
src = open(KERNEL).read()
for tag in ['alloc_ledger.append', 'price_hist.append', 'FLOOR)', 'KAPPA_P',
            'CLEAR=10 if V40_SMOKE else 100', 'np.add.at(contrib', 'mkt_shares=ns']:
    assert tag in src, f"EXCHANGE PLUMBING MISSING: {tag}"
src = src.replace('os.environ.get("V40_SMOKE","1")=="1"', '"0"=="1"')  # FULL ARM flip
assert '"0"=="1"' in src, "smoke->full switch failed"
open(KERNEL, "w").write(src)
print("[assert] plumbing verified; FULL ARM armed (CLEAR=100 follows automatically)")

# ---- 5) wheel probe: real Mamba required (v38e control 2.2706 ran on real Mamba) ----
import torch
print("colab torch:", torch.__version__, "| gpu:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
try:
    from mamba_ssm import Mamba; print("[probe] mamba-ssm ALREADY importable")
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "--no-deps",
      "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0/causal_conv1d-1.7.0+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
      "https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"],
      capture_output=True, text=True)
    try:
        from mamba_ssm import Mamba; print("[probe] wheels OK")
    except Exception as e:
        print("[probe] WHEEL MISMATCH — run: !pip install causal-conv1d==1.7.0 mamba-ssm==2.3.2.post1 --no-build-isolation")
        print("   detail:", str(e)[:200])

# ---- 6) TELEMETRY BEACON: heartbeat log+report to Kaggle every 10 min ----
import shutil, threading, time
_TEL = {"ds": "albanchigozirim/v40-colab-telemetry", "tmp": "/tmp/beacon_push"}
def _beacon_loop():
    while True:
        time.sleep(600)
        try:
            os.makedirs(_TEL["tmp"] + "/v40", exist_ok=True)
            for s, d in [("/content/v40_run.log", "run.log"), ("/content/v40_report.json", "report.json")]:
                if os.path.exists(s): shutil.copy(s, _TEL["tmp"] + "/v40/" + d)
            open(_TEL["tmp"] + "/v40/beat.txt", "w").write(time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()))
            json.dump({"title": "v40-colab-telemetry", "id": _TEL["ds"], "licenses": [{"name": "CC0-1.0"}]},
                      open(_TEL["tmp"] + "/dataset-metadata.json", "w"))
            subprocess.run(["kaggle", "datasets", "version", "-p", _TEL["tmp"], "-m", "beat"], capture_output=True)
        except Exception:
            pass
threading.Thread(target=_beacon_loop, daemon=True).start()
print("[beacon] armed — now run CELL 2 (~3h); first heartbeat lands within 10 min")


In [ ]:
!python /content/data/kernel.py 2>&1 | tee /content/v40_run.log

In [ ]:
import json, math
r = json.load(open('/content/v40_report.json'))
best = r.get('best_val_ce'); ctrl = 2.2706
led = r.get('alloc_ledger', []); ood = r.get('ood_probe', {}) or {}
print('B1 market<=control:', best, 'vs', ctrl, '->', 'PASS' if (best is not None and best<=ctrl) else 'FAIL')
print('B2 OOD:', ood.get('n_distinct'), 'distinct | ood_ok', ood.get('ood_ok'), '->', 'PASS' if (ood.get('ood_ok') and ood.get('n_distinct',0)>=18) else 'FAIL')
print('B3 floor:', 'PASS' if led and min(min(l) for l in led)>=0.049 else 'FAIL', '| clearings:', len(led))
print('B4 supply_resp:', r.get('supply_responsiveness'), '->', 'PASS' if (r.get('supply_responsiveness') or 0)>0.3 else 'FAIL')
print('B5 finite curve:', 'PASS' if all(c.get('val_ce') is not None and math.isfinite(c['val_ce']) for c in r.get('curve',[])) else 'FAIL')
print('alloc final:', r.get('mkt_final'))